In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Carrega a Gold de episodios gerada no notebook 1
df_gold = spark.table("gold_episodios")

print("Total de episodios:", df_gold.count())
print("Colunas:", df_gold.columns)
#df_gold.show(3, truncate=False)

## Feature Engineering

Todas as features comportamentais sao calculadas com dados **ANTERIORES** ao _received_time_ de cada episodio, sem data leakage

In [0]:
# Tabela de transacoes brutas para calculo de historico
df_transacoes = spark.table("silver_events") \
    .filter(F.col("event") == "transaction") \
    .select(
        "account_id",
        F.col("time_since_test_start").alias("transaction_time"),
        "amount"
    )

# Tabela de episodios anteriores para calculo de historico de ofertas
df_episodios_hist = df_gold.select(
    "account_id",
    "offer_id",
    "episodio",
    "received_time",
    "viewed_time",
    "completed_time",
    "offer_type"
)

In [0]:
# Join de cada episodio com transacoes anteriores ao seu received_time
df_feat_compra = df_gold \
    .select("account_id", "offer_id", "episodio", "received_time") \
    .join(df_transacoes, on="account_id", how="left") \
    .filter(F.col("transaction_time") < F.col("received_time")) \
    .groupBy("account_id", "offer_id", "episodio", "received_time") \
    .agg(
        F.count("amount").alias("freq_anterior"),
        F.round(F.avg("amount"), 2).alias("ticket_medio_anterior"),
        F.round(F.sum("amount"), 2).alias("gasto_total_anterior"),
        F.round(F.max("transaction_time"), 2).alias("ultima_transacao_time")
    ) \
    .withColumn(
        "recencia",
        F.round(F.col("received_time") - F.col("ultima_transacao_time"), 2)
    )


In [0]:
# Para cada episodio, conta quantas ofertas anteriores o cliente
# recebeu, visualizou e completou (apenas episodios com received_time menor)

df_hist_ofertas = df_gold \
    .select("account_id", "offer_id", "episodio", "received_time",
            "viewed_time", "completed_time") \
    .alias("atual") \
    .join(
        df_episodios_hist.alias("hist"),
        on=[
            F.col("atual.account_id") == F.col("hist.account_id"),
            F.col("hist.received_time") < F.col("atual.received_time")
        ],
        how="left"
    ) \
    .groupBy(
        F.col("atual.account_id"),
        F.col("atual.offer_id"),
        F.col("atual.episodio"),
        F.col("atual.received_time")
    ) \
    .agg(
        F.count("hist.episodio").alias("n_ofertas_anteriores"),
        F.count(F.when(F.col("hist.viewed_time").isNotNull(), 1)).alias("n_views_anteriores"),
        F.count(F.when(F.col("hist.completed_time").isNotNull(), 1)).alias("n_completadas_anteriores")
    ) \
    .withColumn(
        "taxa_view_historica",
        F.when(
            F.col("n_ofertas_anteriores") > 0,
            F.round(F.col("n_views_anteriores") / F.col("n_ofertas_anteriores"), 3)
        ).otherwise(None)
    ) \
    .withColumn(
        "taxa_conclusao_historica",
        F.when(
            F.col("n_ofertas_anteriores") > 0,
            F.round(F.col("n_completadas_anteriores") / F.col("n_ofertas_anteriores"), 3)
        ).otherwise(None)
    )

In [0]:
df_features = df_gold \
    .join(
        df_feat_compra.drop("received_time"),
        on=["account_id", "offer_id", "episodio"],
        how="left"
    ) \
    .join(
        df_hist_ofertas.drop("received_time"),
        on=["account_id", "offer_id", "episodio"],
        how="left"
    ) \
    .withColumn(
        "ticket_ratio",
        F.when(
            (F.col("min_value") > 0) & (F.col("ticket_medio_anterior").isNotNull()),
            F.round(F.col("ticket_medio_anterior") / F.col("min_value"), 3)
        ).otherwise(None)
    ) \
    .withColumn(
        "days_since_registration",
        F.when(
            F.col("registered_on").isNotNull(),
            F.datediff(
                F.date_add(F.lit("2018-01-01"), F.col("received_day").cast("int")),
                F.col("registered_on")
            )
        ).otherwise(None)
    )

print("Features geradas:", df_features.count(), "| esperado 76277")
print("Colunas:", len(df_features.columns))

# Verifica nulos nas principais features comportamentais
df_features.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ["freq_anterior", "ticket_medio_anterior", "recencia",
              "taxa_view_historica", "taxa_conclusao_historica",
              "ticket_ratio", "days_since_registration"]
]).show()

In [0]:
# ============================================================
# Tratamento de nulos e definição dos targets
# ============================================================

df_model = df_features \
    .withColumn("freq_anterior",
        F.coalesce(F.col("freq_anterior"), F.lit(0))
    ) \
    .withColumn("ticket_medio_anterior",
        F.coalesce(F.col("ticket_medio_anterior"), F.lit(0.0))
    ) \
    .withColumn("gasto_total_anterior",
        F.coalesce(F.col("gasto_total_anterior"), F.lit(0.0))
    ) \
    .withColumn("recencia",
        F.coalesce(F.col("recencia"), F.lit(999.0))
    ) \
    .withColumn("n_ofertas_anteriores",
        F.coalesce(F.col("n_ofertas_anteriores"), F.lit(0))
    ) \
    .withColumn("n_views_anteriores",
        F.coalesce(F.col("n_views_anteriores"), F.lit(0))
    ) \
    .withColumn("n_completadas_anteriores",
        F.coalesce(F.col("n_completadas_anteriores"), F.lit(0))
    ) \
    .withColumn("taxa_view_historica",
        F.coalesce(F.col("taxa_view_historica"), F.lit(0.0))
    ) \
    .withColumn("taxa_conclusao_historica",
        F.coalesce(F.col("taxa_conclusao_historica"), F.lit(0.0))
    ) \
    .withColumn("ticket_ratio",
        F.coalesce(F.col("ticket_ratio"), F.lit(0.0))
    ) \
    .withColumn("gender_encoded",
        F.when(F.col("gender") == "M", 0)
         .when(F.col("gender") == "F", 1)
         .when(F.col("gender") == "O", 2)
         .otherwise(-1)
    ) \
    .withColumn("offer_type_encoded",
        F.when(F.col("offer_type") == "bogo", 0)
         .when(F.col("offer_type") == "discount", 1)
         .when(F.col("offer_type") == "informational", 2)
         .otherwise(-1)
    )

# Target 1: successful_response
# Recebeu + visualizou dentro da validade + completou dentro da validade
# com visualizacao anterior a conclusao
df_model = df_model \
    .withColumn(
        "successful_response",
        F.when(
            (F.col("viewed_time").isNotNull()) &
            (F.col("completed_time").isNotNull()) &
            (F.col("viewed_time") <= F.col("completed_time")),
            1
        ).otherwise(0)
    ) \
    .withColumn(
        "completed_within_validity",
        F.when(F.col("completed_time").isNotNull(), 1).otherwise(0)
    )

# Episodios onde a janela de validade nao foi integralmente observada
# sao excluidos para evitar targets incompletos (censura temporal)
MAX_TIME = 29.75

df_model = df_model \
    .withColumn(
        "censurado",
        F.when(F.col("valid_until") > MAX_TIME, 1).otherwise(0)
    )

# Filtra promocionais e nao censurados
df_model_clean = df_model \
    .filter(F.col("offer_type") != "informational") \
    .filter(F.col("censurado") == 0)

# Divisao temporal por onda de disparo
df_train = df_model_clean.filter(F.col("received_day").isin([0, 7]))
df_val   = df_model_clean.filter(F.col("received_day") == 14)
df_test  = df_model_clean.filter(F.col("received_day") == 17)

print("Dataset completo (promocionais, nao censurados):", df_model_clean.count())
print("Treino (dias 0 e 7):", df_train.count())
print("Validacao (dia 14): ", df_val.count())
print("Teste (dia 17):     ", df_test.count())

print("\nDistribuicao do target successful_response:")
df_model_clean \
    .select("successful_response") \
    .groupBy("successful_response") \
    .count() \
    .orderBy("successful_response") \
    .show()

print("Distribuicao do target completed_within_validity:")
df_model_clean \
    .select("completed_within_validity") \
    .groupBy("completed_within_validity") \
    .count() \
    .orderBy("completed_within_validity") \
    .show()

print("\nEpisodios censurados (excluidos):", df_model.filter(F.col("censurado") == 1).count())

In [0]:
# ============================================================
# Preparacao das features para modelagem
# Conversao de Spark para Pandas para uso com sklearn e LightGBM
# ============================================================

# Lista de features do modelo
FEATURES = [
    # Perfil do cliente
    "age",
    "gender_encoded",
    "credit_card_limit",
    "incomplete_profile",
    "days_since_registration",
    # Comportamento de compra anterior ao envio
    "freq_anterior",
    "ticket_medio_anterior",
    "gasto_total_anterior",
    "recencia",
    # Historico de ofertas anterior ao envio
    "n_ofertas_anteriores",
    "taxa_view_historica",
    "taxa_conclusao_historica",
    # Caracteristicas da oferta
    "offer_type_encoded",
    "discount_value",
    "min_value",
    "duration",
    "n_channels",
    # Interacoes cliente x oferta
    "ticket_ratio"
]

TARGETS = ["successful_response", "completed_within_validity"]

# Seleciona apenas as colunas necessarias antes de converter para pandas
colunas_necessarias = FEATURES + TARGETS + ["received_day"]

train_pd = df_train.select(colunas_necessarias).toPandas()
val_pd   = df_val.select(colunas_necessarias).toPandas()
test_pd  = df_test.select(colunas_necessarias).toPandas()

print("Treino:", train_pd.shape)
print("Validacao:", val_pd.shape)
print("Teste:", test_pd.shape)

print("\nNulos no treino:")
print(train_pd[FEATURES].isnull().sum()[train_pd[FEATURES].isnull().sum() > 0])

print("\nDistribuicao dos targets no treino:")
print("successful_response:", train_pd["successful_response"].value_counts().to_dict())
print("completed_within_validity:", train_pd["completed_within_validity"].value_counts().to_dict())

In [0]:
# Modelo simples para estabelecer um piso de performance
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_auc_score,
    average_precision_score, brier_score_loss
)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")

TARGET = "successful_response"

X_train = train_pd[FEATURES].copy()
y_train = train_pd[TARGET].copy()

X_val = val_pd[FEATURES].copy()
y_val = val_pd[TARGET].copy()

X_test = test_pd[FEATURES].copy()
y_test = test_pd[TARGET].copy()

# Regressao Logistica exige escala padronizada e nao lida com nulos nativamente
# Imputamos a mediana do treino para age e credit_card_limit
for col in ["age", "credit_card_limit"]:
    mediana = X_train[col].median()
    X_train[col] = X_train[col].fillna(mediana)
    X_val[col]   = X_val[col].fillna(mediana)
    X_test[col]  = X_test[col].fillna(mediana)

# Pipeline com padronizacao e regressao logistica
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

pipe_lr.fit(X_train, y_train)

# Avaliacao na validacao e no teste
for nome, X, y in [("Validacao", X_val, y_val), ("Teste", X_test, y_test)]:
    probs = pipe_lr.predict_proba(X)[:, 1]
    preds = pipe_lr.predict(X)
    print(f"\n=== Regressao Logistica — {nome} ===")
    print(f"ROC-AUC:  {roc_auc_score(y, probs):.4f}")
    print(f"PR-AUC:   {average_precision_score(y, probs):.4f}")
    print(f"Brier:    {brier_score_loss(y, probs):.4f}")
    print(classification_report(y, preds, digits=3))

In [0]:
%pip install lightgbm
%pip install shap

In [0]:
# Modelo principal — LightGBM
# Treinado no conjunto de treino, hiperparametros ajustados
# na validacao, avaliado uma unica vez no teste

import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.metrics import classification_report

X_train_lgb = train_pd[FEATURES].copy()
y_train_lgb = train_pd[TARGET].copy()

X_val_lgb = val_pd[FEATURES].copy()
y_val_lgb = val_pd[TARGET].copy()

X_test_lgb = test_pd[FEATURES].copy()
y_test_lgb = test_pd[TARGET].copy()

# LightGBM lida nativamente com nulos, nao precisa de imputacao
model_lgb = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    verbose=-1
)

model_lgb.fit(
    X_train_lgb, y_train_lgb,
    eval_set=[(X_val_lgb, y_val_lgb)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]
)

print("Melhor iteracao:", model_lgb.best_iteration_)

# Avaliacao na validacao e no teste
for nome, X, y in [("Validacao", X_val_lgb, y_val_lgb), ("Teste", X_test_lgb, y_test_lgb)]:
    probs = model_lgb.predict_proba(X)[:, 1]
    preds = model_lgb.predict(X)
    print(f"\n=== LightGBM — {nome} ===")
    print(f"ROC-AUC:  {roc_auc_score(y, probs):.4f}")
    print(f"PR-AUC:   {average_precision_score(y, probs):.4f}")
    print(f"Brier:    {brier_score_loss(y, probs):.4f}")
    print(classification_report(y, preds, digits=3))

In [0]:
# ============================================================
# Calibracao das probabilidades — Platt Scaling
# LightGBM tende a produzir probabilidades extremas (proximas de 0 ou 1)
# A calibracao ajusta as probabilidades para refletir melhor
# a frequencia real de eventos, essencial para o score financeiro
# ============================================================

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

# Calibracao com Platt Scaling usando o conjunto de validacao
# O modelo base ja esta treinado — calibramos so a camada de saida
calibrator = CalibratedClassifierCV(
    model_lgb,
    method="sigmoid",  # Platt Scaling
    cv="prefit"        # usa o modelo ja treinado, nao retreina
)

calibrator.fit(X_val_lgb, y_val_lgb)

# Avaliacao do modelo calibrado no teste
probs_cal = calibrator.predict_proba(X_test_lgb)[:, 1]
probs_raw = model_lgb.predict_proba(X_test_lgb)[:, 1]

print("=== Calibracao — Teste ===")
print(f"Brier Score (sem calibracao): {brier_score_loss(y_test_lgb, probs_raw):.4f}")
print(f"Brier Score (calibrado):      {brier_score_loss(y_test_lgb, probs_cal):.4f}")
print(f"PR-AUC (sem calibracao):      {average_precision_score(y_test_lgb, probs_raw):.4f}")
print(f"PR-AUC (calibrado):           {average_precision_score(y_test_lgb, probs_cal):.4f}")

# Curva de calibracao visual
fig, ax = plt.subplots(figsize=(8, 6))

for probs, label, cor in [
    (probs_raw, "LightGBM sem calibracao", "#E63946"),
    (probs_cal, "LightGBM calibrado (Platt)", "#2A6CF0")
]:
    frac_pos, mean_pred = calibration_curve(y_test_lgb, probs, n_bins=10)
    ax.plot(mean_pred, frac_pos, marker="o", label=label, color=cor)

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Calibracao perfeita")
ax.set_title("Curva de Calibracao — Teste", fontweight="bold")
ax.set_xlabel("Probabilidade prevista")
ax.set_ylabel("Frequencia real de positivos")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

%md
### Calibração das Probabilidades

A curva de calibração mostrou que o LightGBM já produz probabilidades
bem ajustadas para esse dataset — a linha vermelha segue a diagonal de
calibração perfeita com desvio pequeno. A aplicação de Platt Scaling
piorou levemente o Brier Score (0.1773 → 0.1787) sem ganho em PR-AUC.

Decisão: o modelo sem calibração adicional será usado para o score
financeiro. Em produção, recomenda-se reavaliar a calibração com dados
de um período mais longo.

In [0]:
# ============================================================
# Interpretabilidade — SHAP Values
# ============================================================

import shap

# Calcula SHAP values no conjunto de teste
explainer = shap.TreeExplainer(model_lgb)
shap_values = explainer.shap_values(X_test_lgb)

# Para classificacao binaria o LightGBM retorna lista com 2 arrays
# Pegamos o array referente a classe positiva (indice 1)
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

# Summary plot — importancia global das features
plt.figure(figsize=(10, 7))
shap.summary_plot(
    sv, X_test_lgb,
    feature_names=FEATURES,
    plot_type="bar",
    show=False
)
plt.title("Importância das Features — SHAP (LightGBM)", fontweight="bold")
plt.tight_layout()
plt.savefig("/Workspace/Users/matheusmartinez2018@gmail.com/coupon-optimization-ml/data/processed/shap_importance.png",
            dpi=150, bbox_inches="tight")
plt.show()

# Beeswarm plot — distribuicao dos valores SHAP por feature
plt.figure(figsize=(10, 7))
shap.summary_plot(
    sv, X_test_lgb,
    feature_names=FEATURES,
    show=False
)
plt.title("Distribuição dos SHAP Values — Teste", fontweight="bold")
plt.tight_layout()
plt.savefig("/Workspace/Users/matheusmartinez2018@gmail.com/coupon-optimization-ml/data/processed/shap_beeswarm.png",
            dpi=150, bbox_inches="tight")
plt.show()

### Interpretabilidade — SHAP Values

Os SHAP values revelam quais features mais influenciam a probabilidade
de resposta à oferta, e em qual direção.

**Antiguidade na plataforma** (`days_since_registration`) é o sinal mais
forte do modelo. Clientes com mais tempo de conta convertem
significativamente mais — provavelmente porque são mais engajados com
o app e já têm hábito de uso de ofertas.

**Cobertura de canais** (`n_channels`) é a segunda feature mais importante.
Ofertas veiculadas em mais canais (web, email, mobile, social)
alcançam o cliente em mais pontos de contato, aumentando a probabilidade
de resposta. Isso reforça a importância de uma estratégia omnichannel.

**Perfil financeiro** (`credit_card_limit`, `ticket_medio_anterior`, `age`)
concentra as posições 3 a 5. Clientes com maior poder aquisitivo e
ticket histórico mais alto respondem melhor às ofertas — possivelmente
porque têm menor fricção para atingir o valor mínimo de ativação.

**Um achado contraintuitivo confirmado pelo modelo:** `discount_value` tem
impacto negativo — ofertas com desconto maior convertem menos. Isso
é consistente com o que a análise descritiva já havia mostrado e sugere
que o valor do desconto sozinho não é o fator determinante de resposta.
O que importa mais é a acessibilidade da oferta (valor mínimo baixo,
prazo adequado, canais amplos) do que o tamanho do incentivo.

**Cadastros incompletos** (`incomplete_profile`) têm impacto praticamente
nulo no modelo — a ausência de dados demográficos não prediz conversão
além do que as features comportamentais já capturam.

In [0]:
# ============================================================
# Sistema de Recomendação — versão corrigida
# O score financeiro precisa equilibrar probabilidade de resposta
# e valor liquido da oferta. Normalizamos as probabilidades por
# cliente para que a escolha reflita qual oferta aquele cliente
# especifico tem MAIOR CHANCE RELATIVA de responder, ponderada
# pelo valor liquido de cada oferta
# ============================================================

linhas = []
for _, oferta in ofertas_promocionais.iterrows():
    temp = clientes_teste.copy()
    temp["offer_type_encoded"] = 0 if oferta["offer_type"] == "bogo" else 1
    temp["discount_value"]     = oferta["discount_value"]
    temp["min_value"]          = oferta["min_value"]
    temp["duration"]           = oferta["duration"]
    temp["n_channels"]         = oferta["n_channels"]
    temp["ticket_ratio"]       = np.where(
        (temp["ticket_medio_anterior"] > 0) & (oferta["min_value"] > 0),
        temp["ticket_medio_anterior"] / oferta["min_value"],
        0
    )
    temp["offer_id"]       = oferta["offer_id"]
    temp["offer_discount"] = oferta["discount_value"]
    temp["offer_min_value"] = oferta["min_value"]
    linhas.append(temp)

df_rec = pd.concat(linhas, ignore_index=True)

# Probabilidade de resposta para cada par cliente-oferta
df_rec["prob_resposta"] = model_lgb.predict_proba(df_rec[FEATURES])[:, 1]

# Normaliza a probabilidade por cliente: para cada cliente, a oferta
# com maior prob recebe score relativo mais alto independente do desconto
df_rec["prob_min"] = df_rec.groupby("account_id")["prob_resposta"].transform("min")
df_rec["prob_max"] = df_rec.groupby("account_id")["prob_resposta"].transform("max")
df_rec["prob_range"] = df_rec["prob_max"] - df_rec["prob_min"]

df_rec["prob_normalizada"] = np.where(
    df_rec["prob_range"] > 0,
    (df_rec["prob_resposta"] - df_rec["prob_min"]) / df_rec["prob_range"],
    0.5
)

# Score financeiro com probabilidade normalizada:
# P_norm(resposta) * (max(ticket_medio, min_value) - discount_value)
# A normalizacao garante que a probabilidade relativa entre ofertas
# pesa mais do que o valor absoluto do desconto
df_rec["valor_base"] = np.maximum(
    df_rec["ticket_medio_anterior"].clip(lower=0),
    df_rec["offer_min_value"]
)
df_rec["score_financeiro"] = (
    df_rec["prob_normalizada"] * (df_rec["valor_base"] - df_rec["offer_discount"])
)

# Threshold: nao recomenda se todas as probabilidades absolutas sao baixas
# Cliente com prob maxima abaixo de 0.3 recebe "nao enviar"
THRESHOLD_PROB = 0.30

df_rec["nao_enviar"] = df_rec.groupby("account_id")["prob_resposta"].transform("max") < THRESHOLD_PROB

# Melhor oferta por cliente
idx_melhor = df_rec.groupby("account_id")["score_financeiro"].idxmax()
df_recomendacao = df_rec.loc[idx_melhor].copy()
df_recomendacao["recomendacao_final"] = np.where(
    df_recomendacao["nao_enviar"],
    "nao_enviar",
    df_recomendacao["offer_id"]
)
df_recomendacao["offer_type"] = df_recomendacao["offer_type_encoded"].map({0: "bogo", 1: "discount"})

print("Distribuicao das ofertas recomendadas:")
print(df_recomendacao["recomendacao_final"].value_counts())

print("\nDistribuicao por tipo:")
print(df_recomendacao[df_recomendacao["recomendacao_final"] != "nao_enviar"]["offer_type"].value_counts())

print("\nClientes que nao receberiam oferta:", df_recomendacao["nao_enviar"].sum())

print("\nEstatisticas da probabilidade de resposta (melhor oferta por cliente):")
print(df_recomendacao["prob_resposta"].describe().round(3))

In [0]:
# Analise do ranking completo de ofertas por cliente
# Mostra que o modelo discrimina entre ofertas mesmo quando
# a recomendacao final concentra numa so

# Probabilidade media por oferta (media sobre todos os clientes do teste)
print("Probabilidade media de resposta por oferta:")
prob_media = df_rec.groupby("offer_id")["prob_resposta"].mean().sort_values(ascending=False)
print(prob_media.round(3))

# Quantas vezes cada oferta aparece como 1a, 2a, 3a opcao
print("\nRanking das ofertas por posicao:")
df_rec["rank_oferta"] = df_rec.groupby("account_id")["prob_resposta"].rank(
    ascending=False, method="min"
).astype(int)

ranking = df_rec.groupby(["rank_oferta", "offer_id"]).size().unstack(fill_value=0)
print(ranking.head(3))

# Dispersao das probabilidades por cliente
print("\nPara um cliente aleatorio, qual e a diferenca entre")
print("a melhor e a pior oferta em termos de probabilidade?")
spread = df_rec.groupby("account_id")["prob_resposta"].agg(["max", "min"])
spread["diferenca"] = spread["max"] - spread["min"]
print(spread["diferenca"].describe().round(3))

# Visualizacao: distribuicao das probabilidades por oferta
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("Distribuição de Probabilidade de Resposta por Oferta", 
             fontsize=14, fontweight="bold")

for ax, (_, oferta) in zip(axes.flat, ofertas_promocionais.iterrows()):
    subset = df_rec[df_rec["offer_id"] == oferta["offer_id"]]["prob_resposta"]
    ax.hist(subset, bins=30, color="#2A6CF0", edgecolor="white", alpha=0.85)
    ax.axvline(subset.mean(), color="#E63946", linestyle="--", linewidth=1.5)
    ax.set_title(
        f"{oferta['offer_type'].upper()}\ndesc=R${oferta['discount_value']} "
        f"min=R${oferta['min_value']} {int(oferta['duration'])}d",
        fontsize=8, fontweight="bold"
    )
    ax.set_xlabel("P(resposta)", fontsize=7)
    ax.set_ylabel("Clientes", fontsize=7)
    ax.text(0.05, 0.88, f"média={subset.mean():.2f}",
            transform=ax.transAxes, fontsize=7, color="#E63946")

plt.tight_layout()
plt.savefig("/Workspace/Users/matheusmartinez2018@gmail.com/coupon-optimization-ml/data/processed/dist_probabilidades.png",
            dpi=150, bbox_inches="tight")
plt.show()

In [0]:
# ============================================================
# Business Case — Simulação Retrospectiva
# Compara a política histórica observada com a política do modelo
# Apresentado como simulacao, nao como ganho causal comprovado
# ============================================================

# Política histórica: todas as ofertas foram enviadas para todos os clientes
# no conjunto de teste (observado nos dados)
total_episodios_teste = len(test_pd)
total_positivos_teste = test_pd["successful_response"].sum()
taxa_conversao_historica = total_positivos_teste / total_episodios_teste

# Custo total de desconto na politica historica
# Cada episodio que foi completado gerou um custo de discount_value
df_test_spark = df_model_clean.filter(F.col("received_day") == 17)

custo_historico = df_test_spark \
    .filter(F.col("successful_response") == 1) \
    .agg(F.sum("discount_value").alias("custo")).collect()[0]["custo"]

conclusoes_sem_view = df_test_spark \
    .filter(
        (F.col("completed_within_validity") == 1) &
        (F.col("viewed_time").isNull())
    ).count()

print("=== Política Histórica (Teste — Onda dia 17) ===")
print(f"Total de episodios:              {total_episodios_teste:,}")
print(f"Conversoes (successful_response):{total_positivos_teste:,}")
print(f"Taxa de conversao:               {taxa_conversao_historica:.1%}")
print(f"Custo total de desconto:         R$ {custo_historico:,.0f}")
print(f"Conclusoes sem visualizacao:     {conclusoes_sem_view:,}")

# Política do modelo: envia oferta recomendada apenas para clientes
# com prob_resposta acima do threshold
clientes_receberiam = df_recomendacao[~df_recomendacao["nao_enviar"]]
n_envios_modelo = len(clientes_receberiam)
reducao_envios = total_episodios_teste - n_envios_modelo

# Estimativa de conversoes mantidas
# Assumimos que clientes com prob_resposta alta pelo modelo
# teriam respondido na mesma proporcao do historico
conversoes_estimadas = (clientes_receberiam["prob_resposta"] > 0.5).sum()
custo_desconto_modelo = clientes_receberiam["offer_discount"].sum()
economia_desconto = custo_historico - custo_desconto_modelo

print("\n=== Política do Modelo (Simulação Retrospectiva) ===")
print(f"Clientes que receberiam oferta:  {n_envios_modelo:,}")
print(f"Reducao de envios:               {reducao_envios:,} ({reducao_envios/total_episodios_teste:.1%})")
print(f"Custo estimado de desconto:      R$ {custo_desconto_modelo:,.0f}")
print(f"Economia estimada de desconto:   R$ {economia_desconto:,.0f}")
print(f"Reducao de custo:                {economia_desconto/custo_historico:.1%}")

print("\n=== Projecao Anual (assumindo 12 ondas de disparo) ===")
print(f"Economia anual estimada:         R$ {economia_desconto * 12:,.0f}")

print("\nNota: resultados apresentados como simulacao retrospectiva.")
print("Ganhos reais devem ser validados via teste A/B em producao.")

# Tabela de threshold para apresentacao executiva
print("\n=== Tabela de Simulacao por Threshold ===")
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
rows = []
for t in thresholds:
    envios = df_recomendacao[df_recomendacao["prob_resposta"] >= t]
    n_env = len(envios)
    custo = envios["offer_discount"].sum()
    rows.append({
        "Threshold": t,
        "Envios": n_env,
        "Reducao_envios_%": f"{(total_episodios_teste - n_env)/total_episodios_teste:.1%}",
        "Custo_desconto": f"R$ {custo:,.0f}",
        "Economia_vs_historico": f"R$ {custo_historico - custo:,.0f}"
    })

import pandas as pd
df_threshold = pd.DataFrame(rows)
print(df_threshold.to_string(index=False))

%md
## Business Case — Simulação Retrospectiva

A análise foi conduzida sobre a onda de teste (dia 17), com 10.210
episódios de oferta e taxa histórica de conversão de 38,6%.

**Conclusões sem visualização registrada**

1.135 episódios foram concluídos sem que o cliente tivesse visualizado
a oferta dentro da janela de validade. Esses casos representam possível
subsídio desnecessário — o desconto foi pago sem evidência de que a
oferta influenciou a decisão de compra. Não é possível afirmar com
certeza que essas compras teriam ocorrido sem o incentivo, mas são
candidatos prioritários para redução de custo em produção.

**Simulação por threshold de probabilidade**

O modelo permite controlar o equilíbrio entre cobertura e custo via
threshold de probabilidade mínima para envio:

| Threshold | Envios | Redução | Economia por onda | Economia anual est. |
|---|---|---|---|---|
| 0.3 | 9.368 | 8% | R$ 302 | R$ 3.624 |
| 0.4 | 8.493 | 17% | R$ 2.060 | R$ 24.720 |
| 0.5 | 7.456 | 27% | R$ 4.155 | R$ 49.860 |
| 0.6 | 6.235 | 39% | R$ 6.680 | R$ 80.160 |
| 0.7 | 4.796 | 53% | R$ 9.733 | R$ 116.796 |

A escolha do threshold depende do apetite de risco do negócio — um
threshold mais alto economiza mais desconto mas reduz a cobertura de
clientes potencialmente conversíveis. A validação do ponto ótimo deve
ser feita via teste A/B em produção, comparando grupos com thresholds
diferentes sobre uma base de clientes real.

**Limitações da simulação**

Os resultados acima são uma simulação retrospectiva sobre dados
históricos, não um experimento controlado. Não é possível afirmar com
certeza que clientes que não receberiam oferta pelo modelo teriam de
fato convertido — nem o contrário. O valor real da estratégia só pode
ser mensurado com precisão via teste A/B prospectivo.

In [0]:
# ============================================================
# Lift@K — avalia a qualidade do ranking do modelo
# Mede quantas vezes o modelo e melhor do que enviar aleatoriamente
# Util para justificar o valor do ranking, nao so da classificacao
# ============================================================

from sklearn.metrics import roc_curve

# Probabilidades e targets reais do conjunto de teste
probs_teste = model_lgb.predict_proba(X_test_lgb)[:, 1]
y_teste = y_test_lgb.values

# Ordena por probabilidade decrescente
idx_ord = np.argsort(probs_teste)[::-1]
y_ord = y_teste[idx_ord]

# Calcula lift para cada percentil
n = len(y_teste)
taxa_positivos_global = y_teste.mean()

ks_values = []
lift_values = []
percentis = np.arange(0.01, 1.01, 0.01)

conversoes_acumuladas = np.cumsum(y_ord)
conversoes_aleatorio = taxa_positivos_global * np.arange(1, n + 1)

for p in percentis:
    k = int(p * n)
    if k == 0:
        continue
    lift = (conversoes_acumuladas[k-1] / k) / taxa_positivos_global
    lift_values.append(lift)

# KS statistic
fpr, tpr, _ = roc_curve(y_teste, probs_teste)
ks_stat = np.max(tpr - fpr)

print(f"KS Statistic: {ks_stat:.4f}")
print(f"Lift@10%: {lift_values[9]:.2f}x")
print(f"Lift@20%: {lift_values[19]:.2f}x")
print(f"Lift@30%: {lift_values[29]:.2f}x")
print(f"Lift@50%: {lift_values[49]:.2f}x")

# ── Graficos finais de avaliacao ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Avaliação do Modelo — LightGBM", fontsize=14, fontweight="bold")

# 1. Curva de Lift
ax1 = axes[0]
ax1.plot(percentis[:len(lift_values)], lift_values,
         color="#2A6CF0", linewidth=2, label="LightGBM")
ax1.axhline(1, color="gray", linestyle="--", linewidth=1, label="Baseline aleatório")
ax1.fill_between(percentis[:len(lift_values)], 1, lift_values,
                 alpha=0.15, color="#2A6CF0")
ax1.set_title("Curva de Lift", fontweight="bold")
ax1.set_xlabel("Percentil da população contatada")
ax1.set_ylabel("Lift (vezes melhor que aleatório)")
ax1.legend()
ax1.grid(alpha=0.3)

# 2. Curva de Ganho Acumulado
ax2 = axes[1]
ganho_modelo = conversoes_acumuladas / conversoes_acumuladas[-1]
ganho_aleatorio = np.linspace(0, 1, n)
ax2.plot(np.linspace(0, 1, n), ganho_modelo,
         color="#2A6CF0", linewidth=2, label="LightGBM")
ax2.plot([0, 1], [0, 1], color="gray", linestyle="--",
         linewidth=1, label="Baseline aleatório")
ax2.plot([0, taxa_positivos_global, 1], [0, 1, 1],
         color="#E63946", linestyle="--", linewidth=1, label="Modelo perfeito")
ax2.set_title("Curva de Ganho Acumulado", fontweight="bold")
ax2.set_xlabel("Fração da população contatada")
ax2.set_ylabel("Fração das conversões capturadas")
ax2.legend()
ax2.grid(alpha=0.3)

# 3. Distribuicao de probabilidades por classe
ax3 = axes[2]
ax3.hist(probs_teste[y_teste == 0], bins=40, alpha=0.6,
         color="#E63946", label="Não converteu", density=True)
ax3.hist(probs_teste[y_teste == 1], bins=40, alpha=0.6,
         color="#2A6CF0", label="Converteu", density=True)
ax3.axvline(0.5, color="gray", linestyle="--", linewidth=1)
ax3.set_title("Distribuição de Probabilidades\npor Classe Real", fontweight="bold")
ax3.set_xlabel("P(resposta)")
ax3.set_ylabel("Densidade")
ax3.legend()
ax3.grid(alpha=0.3)

plt.tight_layout()
#plt.savefig("/Workspace/Users/matheusmartinez2018@gmail.com/coupon-optimization-ml/data/processed/avaliacao_modelo.#png",
#            dpi=150, bbox_inches="tight")
plt.show()
#print("Figura salva.")

%md
## Avaliação do Modelo — LightGBM

| Métrica | Regressão Logística | LightGBM | 
|---|---|---|
| ROC-AUC | 0.752 | 0.799 |
| PR-AUC | 0.633 | 0.685 |
| Brier Score | 0.205 | 0.177 |
| F1 (positivo) | 0.434 | 0.644 |
| KS Statistic | - | 0.456 |

O LightGBM supera a Regressão Logística em todas as métricas.
O ganho mais expressivo é no F1 da classe positiva (+0.21), o que
significa que o modelo identifica quem vai converter com muito mais
qualidade — essencial para um sistema de recomendação onde falsos
negativos têm custo direto (conversões perdidas).

**Curva de Lift:** contatando os 10% de clientes com maior
probabilidade prevista, o modelo captura 2x mais conversões do que
um envio aleatório. Aos 30%, ainda mantém lift de 1.77x.

**Curva de Ganho Acumulado:** contatando 40% dos clientes rankeados
pelo modelo, captura-se aproximadamente 70% de todas as conversões
do período. Isso fundamenta a estratégia de segmentação: concentrar
esforço e custo de desconto nos clientes de maior propensão.

In [0]:
# ============================================================
# Correção do leakage nas features históricas
# A versão anterior verificava apenas que o episodio historico
# foi RECEBIDO antes do atual. Isso nao garante que a visualizacao
# ou conclusao desse episodio ja havia ocorrido no momento do envio.
# Aqui corrigimos: so contamos viewed/completed que ja ocorreram
# antes do received_time do episodio atual
# ============================================================

# Features de comportamento de compra — sem mudanca (ja estava correto)
df_transacoes = spark.table("silver_events") \
    .filter(F.col("event") == "transaction") \
    .select(
        "account_id",
        F.col("time_since_test_start").alias("transaction_time"),
        "amount"
    )

df_feat_compra = df_gold \
    .select("account_id", "offer_id", "episodio", "received_time") \
    .join(df_transacoes, on="account_id", how="left") \
    .filter(F.col("transaction_time") < F.col("received_time")) \
    .groupBy("account_id", "offer_id", "episodio", "received_time") \
    .agg(
        F.count("amount").alias("freq_anterior"),
        F.round(F.avg("amount"), 2).alias("ticket_medio_anterior"),
        F.round(F.sum("amount"), 2).alias("gasto_total_anterior"),
        F.round(F.max("transaction_time"), 2).alias("ultima_transacao_time")
    ) \
    .withColumn(
        "recencia",
        F.round(F.col("received_time") - F.col("ultima_transacao_time"), 2)
    )

# Features de historico promocional — CORRIGIDO
# Versao anterior: contava episodios onde received_time < atual.received_time
# mas nao verificava se o viewed_time/completed_time ja tinha ocorrido
# Versao corrigida: viewed e completed so contam se ocorreram
# ANTES do received_time do episodio atual
df_episodios_hist = df_gold.select(
    "account_id", "offer_id", "episodio",
    "received_time", "viewed_time", "completed_time", "offer_type"
)

df_hist_ofertas = df_gold \
    .select("account_id", "offer_id", "episodio", "received_time",
            "offer_type") \
    .alias("atual") \
    .join(
        df_episodios_hist.alias("hist"),
        on=[
            F.col("atual.account_id") == F.col("hist.account_id"),
            F.col("hist.received_time") < F.col("atual.received_time")
        ],
        how="left"
    ) \
    .groupBy(
        F.col("atual.account_id"),
        F.col("atual.offer_id"),
        F.col("atual.episodio"),
        F.col("atual.received_time"),
        F.col("atual.offer_type")
    ) \
    .agg(
        F.count("hist.episodio").alias("n_ofertas_anteriores"),
        # viewed conta so se viewed_time < received_time atual
        F.count(
            F.when(
                F.col("hist.viewed_time").isNotNull() &
                (F.col("hist.viewed_time") < F.col("atual.received_time")),
                1
            )
        ).alias("n_views_anteriores"),
        # completed conta so se completed_time < received_time atual
        F.count(
            F.when(
                F.col("hist.completed_time").isNotNull() &
                (F.col("hist.completed_time") < F.col("atual.received_time")),
                1
            )
        ).alias("n_completadas_anteriores"),
        # historico por tipo de oferta — corrigido da mesma forma
        F.count(
            F.when(
                (F.col("hist.offer_type") == "bogo") &
                F.col("hist.completed_time").isNotNull() &
                (F.col("hist.completed_time") < F.col("atual.received_time")),
                1
            )
        ).alias("n_completadas_bogo_anterior"),
        F.count(
            F.when(
                (F.col("hist.offer_type") == "discount") &
                F.col("hist.completed_time").isNotNull() &
                (F.col("hist.completed_time") < F.col("atual.received_time")),
                1
            )
        ).alias("n_completadas_discount_anterior")
    ) \
    .withColumn(
        "taxa_view_historica",
        F.when(
            F.col("n_ofertas_anteriores") > 0,
            F.round(F.col("n_views_anteriores") / F.col("n_ofertas_anteriores"), 3)
        ).otherwise(None)
    ) \
    .withColumn(
        "taxa_conclusao_historica",
        F.when(
            F.col("n_ofertas_anteriores") > 0,
            F.round(F.col("n_completadas_anteriores") / F.col("n_ofertas_anteriores"), 3)
        ).otherwise(None)
    )

# Monta dataset completo com features corrigidas
df_features_v2 = df_gold \
    .join(
        df_feat_compra.drop("received_time"),
        on=["account_id", "offer_id", "episodio"],
        how="left"
    ) \
    .join(
        df_hist_ofertas.drop("received_time", "offer_type"),
        on=["account_id", "offer_id", "episodio"],
        how="left"
    ) \
    .withColumn(
        "ticket_ratio",
        F.when(
            (F.col("min_value") > 0) & (F.col("ticket_medio_anterior").isNotNull()),
            F.round(F.col("ticket_medio_anterior") / F.col("min_value"), 3)
        ).otherwise(None)
    ) \
    .withColumn(
        "discount_ratio",
        F.when(
            F.col("min_value") > 0,
            F.round(F.col("discount_value") / F.col("min_value"), 3)
        ).otherwise(None)
    ) \
    .withColumn(
        "ticket_minus_minvalue",
        F.when(
            F.col("ticket_medio_anterior").isNotNull(),
            F.round(F.col("ticket_medio_anterior") - F.col("min_value"), 2)
        ).otherwise(None)
    ) \
    .withColumn(
    "days_since_registration",
    F.when(
        F.col("registered_on").isNotNull(),
        F.datediff(
            F.date_add(F.lit("2018-07-26"), F.col("received_day").cast("int")),
            F.col("registered_on")
        )
    ).otherwise(None)
)

print("Features v2 geradas:", df_features_v2.count(), "| esperado 76277")

In [0]:
# ============================================================
# Tratamento de nulos, targets, split e re-treinamento
# ============================================================

FEATURES_V2 = [
    # Perfil do cliente
    "age",
    "gender_encoded",
    "credit_card_limit",
    "incomplete_profile",
    "days_since_registration",
    # Comportamento de compra anterior
    "freq_anterior",
    "ticket_medio_anterior",
    "gasto_total_anterior",
    "recencia",
    # Historico promocional corrigido
    "n_ofertas_anteriores",
    "n_views_anteriores",
    "n_completadas_anteriores",
    "n_completadas_bogo_anterior",
    "n_completadas_discount_anterior",
    "taxa_view_historica",
    "taxa_conclusao_historica",
    # Caracteristicas da oferta
    "offer_type_encoded",
    "discount_value",
    "min_value",
    "duration",
    "n_channels",
    # Interacoes cliente x oferta
    "ticket_ratio",
    "discount_ratio",
    "ticket_minus_minvalue"
]

TARGET = "successful_response"

df_model_v2 = df_features_v2 \
    .withColumn("freq_anterior", F.coalesce(F.col("freq_anterior"), F.lit(0))) \
    .withColumn("ticket_medio_anterior", F.coalesce(F.col("ticket_medio_anterior"), F.lit(0.0))) \
    .withColumn("gasto_total_anterior", F.coalesce(F.col("gasto_total_anterior"), F.lit(0.0))) \
    .withColumn("recencia", F.coalesce(F.col("recencia"), F.lit(999.0))) \
    .withColumn("n_ofertas_anteriores", F.coalesce(F.col("n_ofertas_anteriores"), F.lit(0))) \
    .withColumn("n_views_anteriores", F.coalesce(F.col("n_views_anteriores"), F.lit(0))) \
    .withColumn("n_completadas_anteriores", F.coalesce(F.col("n_completadas_anteriores"), F.lit(0))) \
    .withColumn("n_completadas_bogo_anterior", F.coalesce(F.col("n_completadas_bogo_anterior"), F.lit(0))) \
    .withColumn("n_completadas_discount_anterior", F.coalesce(F.col("n_completadas_discount_anterior"), F.lit(0))) \
    .withColumn("taxa_view_historica", F.coalesce(F.col("taxa_view_historica"), F.lit(0.0))) \
    .withColumn("taxa_conclusao_historica", F.coalesce(F.col("taxa_conclusao_historica"), F.lit(0.0))) \
    .withColumn("ticket_ratio", F.coalesce(F.col("ticket_ratio"), F.lit(0.0))) \
    .withColumn("discount_ratio", F.coalesce(F.col("discount_ratio"), F.lit(0.0))) \
    .withColumn("ticket_minus_minvalue", F.coalesce(F.col("ticket_minus_minvalue"), F.lit(0.0))) \
    .withColumn("gender_encoded",
        F.when(F.col("gender") == "M", 0)
         .when(F.col("gender") == "F", 1)
         .when(F.col("gender") == "O", 2)
         .otherwise(-1)
    ) \
    .withColumn("offer_type_encoded",
        F.when(F.col("offer_type") == "bogo", 0)
         .when(F.col("offer_type") == "discount", 1)
         .otherwise(-1)
    ) \
    .withColumn(
        "successful_response",
        F.when(
            F.col("viewed_time").isNotNull() &
            F.col("completed_time").isNotNull() &
            (F.col("viewed_time") <= F.col("completed_time")),
            1
        ).otherwise(0)
    ) \
    .withColumn(
        "completed_within_validity",
        F.when(F.col("completed_time").isNotNull(), 1).otherwise(0)
    ) \
    .withColumn(
        "censurado",
        F.when(F.col("valid_until") > 29.75, 1).otherwise(0)
    )

df_model_clean_v2 = df_model_v2 \
    .filter(F.col("offer_type") != "informational") \
    .filter(F.col("censurado") == 0)

colunas_pd = ["account_id"] + FEATURES_V2 + ["successful_response", "completed_within_validity", "received_day"]

train_pd_v2 = df_model_clean_v2.filter(F.col("received_day").isin([0, 7])).select(colunas_pd).toPandas()
val_pd_v2   = df_model_clean_v2.filter(F.col("received_day") == 14).select(colunas_pd).toPandas()
test_pd_v2  = df_model_clean_v2.filter(F.col("received_day") == 17).select(colunas_pd).toPandas()

print("Treino:", train_pd_v2.shape)
print("Validacao:", val_pd_v2.shape)
print("Teste:", test_pd_v2.shape)

X_train_v2 = train_pd_v2[FEATURES_V2]
y_train_v2 = train_pd_v2[TARGET]
X_val_v2   = val_pd_v2[FEATURES_V2]
y_val_v2   = val_pd_v2[TARGET]
X_test_v2  = test_pd_v2[FEATURES_V2]
y_test_v2  = test_pd_v2[TARGET]

# Re-treina LightGBM com features corrigidas
model_v2 = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    verbose=-1
)

model_v2.fit(
    X_train_v2, y_train_v2,
    eval_set=[(X_val_v2, y_val_v2)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]
)

# Calcula metricas para comparacao antes x depois
def avaliar(model, X, y, nome):
    probs = model.predict_proba(X)[:, 1]
    preds = model.predict(X)
    n = len(y)
    idx_ord = np.argsort(probs)[::-1]
    y_ord = np.array(y)[idx_ord]
    taxa_global = y.mean()
    lift10 = (y_ord[:int(0.1*n)].mean()) / taxa_global
    lift20 = (y_ord[:int(0.2*n)].mean()) / taxa_global
    return {
        "Conjunto": nome,
        "ROC-AUC": round(roc_auc_score(y, probs), 4),
        "PR-AUC": round(average_precision_score(y, probs), 4),
        "Brier": round(brier_score_loss(y, probs), 4),
        "Lift@10%": round(lift10, 2),
        "Lift@20%": round(lift20, 2)
    }

# Modelo v1 (com leakage) — usa variaveis do escopo anterior se ainda existirem
resultados = []
try:
    resultados.append(avaliar(model_lgb, X_val_lgb, y_val_lgb, "v1 — Validacao (com leakage)"))
    resultados.append(avaliar(model_lgb, X_test_lgb, y_test_lgb, "v1 — Teste (com leakage)"))
except:
    print("Modelo v1 nao disponivel na sessao — comparacao sera so com v2")

resultados.append(avaliar(model_v2, X_val_v2, y_val_v2, "v2 — Validacao (corrigido)"))
resultados.append(avaliar(model_v2, X_test_v2, y_test_v2, "v2 — Teste (corrigido)"))

print("\n=== Comparacao antes x depois da correcao do leakage ===")
print(pd.DataFrame(resultados).to_string(index=False))

In [0]:
# ============================================================
# Baseline: melhor oferta para todos
# Compara a recomendacao individual do modelo contra a estrategia
# simples de enviar a oferta historicamente mais eficiente para todos
# ============================================================

# Taxa de resposta historica por oferta no treino
taxa_por_oferta_treino = train_pd_v2.groupby("offer_type_encoded")["successful_response"].mean()

# Identifica a oferta dominante por offer_type_encoded no treino
# (no nosso caso o modelo ja mostrou que discount domina)
print("=== Taxa historica de resposta por tipo no treino ===")
print(taxa_por_oferta_treino.round(3))

# Baseline 1: envia a mesma oferta para todos (a de maior taxa historica)
melhor_tipo_treino = taxa_por_oferta_treino.idxmax()
baseline_preds = np.ones(len(y_test_v2)) * (taxa_por_oferta_treino[melhor_tipo_treino])

print("\n=== Comparacao: LightGBM v2 vs Baseline melhor oferta global ===")
probs_v2_teste = model_v2.predict_proba(X_test_v2)[:, 1]

print(f"Baseline (melhor oferta para todos):")
print(f"  PR-AUC:   {average_precision_score(y_test_v2, baseline_preds):.4f}")
print(f"  Brier:    {brier_score_loss(y_test_v2, baseline_preds):.4f}")

print(f"\nLightGBM v2:")
print(f"  ROC-AUC:  {roc_auc_score(y_test_v2, probs_v2_teste):.4f}")
print(f"  PR-AUC:   {average_precision_score(y_test_v2, probs_v2_teste):.4f}")
print(f"  Brier:    {brier_score_loss(y_test_v2, probs_v2_teste):.4f}")

# ============================================================
# Business Case Corrigido
# Usa reward real observado, nao discount_value estimado
# Custo esperado = prob_resposta * discount_value (nao custo integral)
# Conversoes estimadas = soma das probabilidades (nao contagem > 0.5)
# ============================================================

# Recarrega dados do teste com reward real
df_teste_spark = df_model_clean_v2.filter(F.col("received_day") == 17)

reward_historico = df_teste_spark \
    .filter(F.col("completed_within_validity") == 1) \
    .agg(F.sum("reward").alias("reward_total")) \
    .collect()[0]["reward_total"]

total_episodios = df_teste_spark.count()
total_conversoes = df_teste_spark.filter(F.col("successful_response") == 1).count()
conclusoes_sem_view = df_teste_spark \
    .filter(
        (F.col("completed_within_validity") == 1) &
        (F.col("viewed_time").isNull())
    ).count()

print("\n=== Política Histórica — Onda Teste (dia 17) ===")
print(f"Total de episodios:             {total_episodios:,}")
print(f"Conversoes (successful_resp):   {total_conversoes:,}  ({total_conversoes/total_episodios:.1%})")
print(f"Reward total pago:              R$ {reward_historico:,.0f}")
print(f"Conclusoes sem visualizacao:    {conclusoes_sem_view:,}  ({conclusoes_sem_view/total_episodios:.1%})")

test_pd_v2_rec = test_pd_v2.copy()
test_pd_v2_rec["prob_v2"] = probs_v2_teste
test_pd_v2_rec["custo_esperado"] = test_pd_v2_rec["prob_v2"] * test_pd_v2_rec["discount_value"]

print("\n=== Tabela de Simulacao por Threshold ===")
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
rows = []
for t in thresholds:
    selecionados = test_pd_v2_rec[test_pd_v2_rec["prob_v2"] >= t]
    n_sel = len(selecionados)
    conv_estimadas = selecionados["prob_v2"].sum()
    custo_esperado = selecionados["custo_esperado"].sum()
    conv_observadas = selecionados["successful_response"].sum()
    pct_conv_preservadas = conv_observadas / total_conversoes if total_conversoes > 0 else 0
    taxa_resp_selecionados = conv_observadas / n_sel if n_sel > 0 else 0

    rows.append({
        "Threshold": t,
        "Selecionados": n_sel,
        "Reducao_%": f"{(total_episodios - n_sel)/total_episodios:.1%}",
        "Conv_observadas": int(conv_observadas),
        "Conv_preservadas_%": f"{pct_conv_preservadas:.1%}",
        "Taxa_resp_%": f"{taxa_resp_selecionados:.1%}",
        "Custo_esperado": f"R$ {custo_esperado:,.0f}",
        "Pot_economia": f"R$ {reward_historico - custo_esperado:,.0f}"
    })

print(pd.DataFrame(rows).to_string(index=False))
print("\nNota: simulacao retrospectiva. Custo esperado = prob * discount_value.")
print("Ganhos reais devem ser validados via teste A/B em producao.")

In [0]:
# ============================================================
# Validacao da cobertura de ofertas por split
# ============================================================

# Puxa direto do Spark onde offer_id esta disponivel
colunas_cob = ["offer_id", "offer_type", "received_day", "successful_response"]

cob_spark = df_model_clean_v2 \
    .select(colunas_cob) \
    .withColumn("split",
        F.when(F.col("received_day").isin([0, 7]), "treino")
         .when(F.col("received_day") == 14, "validacao")
         .when(F.col("received_day") == 17, "teste")
         .otherwise("outro")
    ) \
    .filter(F.col("split") != "outro") \
    .groupBy("split", "offer_id", "offer_type") \
    .agg(
        F.count("*").alias("n_episodios"),
        F.round(F.avg("successful_response"), 3).alias("taxa_resposta")
    ) \
    .orderBy("offer_id", "split")

print("=== Cobertura de ofertas por split ===")
cob_spark.show(40, truncate=False)

# Diagnostico
ofertas_por_split = cob_spark.groupBy("split") \
    .agg(F.collect_set("offer_id").alias("ofertas"))

ofertas_treino = set(cob_spark.filter(F.col("split") == "treino")
                    .select("offer_id").distinct().toPandas()["offer_id"])
ofertas_val    = set(cob_spark.filter(F.col("split") == "validacao")
                    .select("offer_id").distinct().toPandas()["offer_id"])
ofertas_teste  = set(cob_spark.filter(F.col("split") == "teste")
                    .select("offer_id").distinct().toPandas()["offer_id"])

print(f"Ofertas no treino:    {len(ofertas_treino)}")
print(f"Ofertas na validacao: {len(ofertas_val)}")
print(f"Ofertas no teste:     {len(ofertas_teste)}")

so_val  = ofertas_val - ofertas_treino
so_test = ofertas_teste - ofertas_treino
print(f"\nOfertas APENAS na validacao: {so_val if so_val else 'nenhuma'}")
print(f"Ofertas APENAS no teste:     {so_test if so_test else 'nenhuma'}")


%md
### Validação da Cobertura de Ofertas por Amostra

Todas as 8 ofertas promocionais aparecem nos três conjuntos com volumes
equilibrados (entre 1.222 e 1.335 episódios por oferta no teste) e
taxas de resposta estáveis entre splits. Não há oferta exclusiva de
validação ou teste que o modelo nunca tenha visto no treino.

As taxas de resposta são consistentes ao longo das ondas — variação
máxima de 3 pontos percentuais entre treino e teste para a mesma
oferta — confirmando que o comportamento das campanhas é estável no
período analisado e que a divisão temporal não introduziu viés de
seleção.

O modelo não usa `offer_id` diretamente como feature. As ofertas são
representadas por seus atributos (`offer_type`, `discount_value`,
`min_value`, `duration`, `n_channels`), o que permite que o modelo generaliza para configurações semelhantes às observadas no treino.

In [0]:
# ============================================================
# Verificacao do criterio de censura temporal
# Confirma que nao existe corte fixo no codigo e que o criterio
# utilizado e valid_until <= 29.75 (received_time + duration)
# ============================================================

MAX_TIME = 29.75

# Distribuicao dos episodios censurados por offer_id e duracao
df_censura = df_model_v2 \
    .filter(F.col("offer_type") != "informational") \
    .withColumn(
        "valido",
        F.when(F.col("valid_until") <= MAX_TIME, "nao_censurado").otherwise("censurado")
    ) \
    .groupBy("offer_id", "offer_type", "duration", "valido") \
    .agg(
        F.count("*").alias("n_episodios"),
        F.round(F.avg("successful_response"), 3).alias("taxa_resposta")
    ) \
    .orderBy("offer_id", "valido")

print("=== Episodios por oferta, duracao e status de censura ===")
df_censura.show(40, truncate=False)

# Totais
total_validos   = df_model_v2 \
    .filter(F.col("offer_type") != "informational") \
    .filter(F.col("valid_until") <= MAX_TIME).count()
total_censurados = df_model_v2 \
    .filter(F.col("offer_type") != "informational") \
    .filter(F.col("valid_until") > MAX_TIME).count()

print(f"Total nao censurados (valid_until <= 29.75): {total_validos:,}")
print(f"Total censurados (valid_until > 29.75):      {total_censurados:,}")
print(f"\nCriterio utilizado: received_time + duration <= {MAX_TIME}")
print("Nao existe corte fixo em 26.5 no codigo — criterio correto ja estava implementado.")

%md
### Validação do Critério de Censura Temporal

O critério utilizado é `valid_until <= 29.75`, equivalente a
`received_time + duration <= 29.75`. Não existe corte fixo arbitrário
no código — cada episódio é avaliado conforme a duração real da oferta.

**Episódios excluídos por oferta:**

Ofertas de 10 dias (`0b1e1539` e `fafdcd668`) perderam ~33% dos
episódios por censura. Ofertas de 7 dias perderam ~17%. Ofertas de 5
dias (`4d5c57ea` e `f19421c1`) não tiveram nenhum episódio censurado,
pois mesmo recebidas na última onda (dia 24) ainda têm janela completa
observável até o dia 29.

**Por que a exclusão é necessária:**

As taxas de resposta dos episódios censurados são consistentemente
menores do que as dos não censurados para a mesma oferta — por exemplo,
`fafdcd668` censurada converte 54.7% vs 63.1% não censurada. Incluir
esses episódios classificaria conversões ainda pendentes como
`successful_response = 0`, introduzindo falsos negativos sistemáticos
que enviasariam o modelo contra envios tardios.

**Total excluído:** 10.236 episódios (16.8% do total promocional).
**Dataset final para modelagem:** 50.806 episódios.

In [0]:
# ============================================================
# Business Case Revisado — considerando receita das transacoes
# O valor de negocio nao e so economizar desconto, mas tambem
# capturar a receita das transacoes que as ofertas geram
# ============================================================

# Ticket medio real das transacoes associadas a conversoes no teste
# Usamos a Silver para pegar o valor real das transacoes
df_transacoes_teste = spark.table("silver_events") \
    .filter(F.col("event") == "transaction") \
    .join(
        df_model_clean_v2
            .filter(F.col("received_day") == 17)
            .filter(F.col("successful_response") == 1)
            .select("account_id",
                    F.col("completed_time").alias("comp_time"),
                    F.col("received_time").alias("recv_time"),
                    "valid_until", "discount_value", "reward"),
        on="account_id",
        how="inner"
    ) \
    .filter(
        (F.col("time_since_test_start") >= F.col("recv_time")) &
        (F.col("time_since_test_start") <= F.col("valid_until"))
    )

ticket_stats = df_transacoes_teste.agg(
    F.count("amount").alias("n_transacoes"),
    F.round(F.avg("amount"), 2).alias("ticket_medio"),
    F.round(F.sum("amount"), 2).alias("receita_total"),
    F.round(F.percentile_approx("amount", 0.5), 2).alias("ticket_mediana")
).collect()[0]

print("=== Transacoes associadas a conversoes no teste ===")
print(f"N transacoes:    {ticket_stats['n_transacoes']:,}")
print(f"Ticket medio:    R$ {ticket_stats['ticket_medio']:.2f}")
print(f"Ticket mediana:  R$ {ticket_stats['ticket_mediana']:.2f}")
print(f"Receita total:   R$ {ticket_stats['receita_total']:,.0f}")

ticket_medio = ticket_stats["ticket_medio"]
receita_total = ticket_stats["receita_total"]
reward_historico = 29398

print("\n=== Business Case Completo — Onda Teste (dia 17) ===")
print(f"Receita total das transacoes:   R$ {receita_total:,.0f}")
print(f"Reward total pago:              R$ {reward_historico:,.0f}")
print(f"Receita liquida historica:      R$ {receita_total - reward_historico:,.0f}")
print(f"Margem de desconto:             {reward_historico/receita_total:.1%}")

# Simulacao revisada por threshold
print("\n=== Tabela Revisada por Threshold ===")
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
rows = []
for t in thresholds:
    selecionados = test_pd_v2_rec[test_pd_v2_rec["prob_v2"] >= t]
    n_sel = len(selecionados)
    conv_obs = selecionados["successful_response"].sum()
    custo_esp = selecionados["custo_esperado"].sum()
    receita_estimada = conv_obs * ticket_medio
    valor_liquido = receita_estimada - custo_esp
    pct_conv = conv_obs / len(test_pd_v2_rec[test_pd_v2_rec["successful_response"] == 1])

    rows.append({
        "Threshold": t,
        "Selecionados": n_sel,
        "Reducao_%": f"{(10210-n_sel)/10210:.1%}",
        "Conv_obs": int(conv_obs),
        "Conv_pres_%": f"{pct_conv:.1%}",
        "Receita_est": f"R$ {receita_estimada:,.0f}",
        "Custo_esp": f"R$ {custo_esp:,.0f}",
        "Valor_liquido": f"R$ {valor_liquido:,.0f}"
    })

print(pd.DataFrame(rows).to_string(index=False))

print(f"\nReferencia historica:")
print(f"  Receita:       R$ {receita_total:,.0f}")
print(f"  Custo reward:  R$ {reward_historico:,.0f}")
print(f"  Valor liquido: R$ {receita_total - reward_historico:,.0f}")
print("\nNota: receita estimada = conversoes observadas x ticket medio real.")
print("Simulacao retrospectiva. Validar via A/B test em producao.")

In [0]:
# ============================================================
# Analise de publico por faixa de propensao
# Agrupa clientes por probabilidade prevista em vez de oferta
# recomendada, porque a concentracao em uma oferta nao permite
# diferenciar perfis por campanha de forma acionavel
# ============================================================

# Reconstroi df_rec com features de perfil para analise
# Usa test_pd_v2 que ja tem todas as features
test_pd_v2_rec = test_pd_v2.copy()
test_pd_v2_rec["prob_v2"] = model_v2.predict_proba(X_test_v2)[:, 1]
test_pd_v2_rec["custo_esperado"] = test_pd_v2_rec["prob_v2"] * test_pd_v2_rec["discount_value"]
test_pd_v2_rec["valor_base"] = np.maximum(
    test_pd_v2_rec["ticket_medio_anterior"].clip(lower=0),
    test_pd_v2_rec["min_value"]
)
test_pd_v2_rec["valor_liquido_est"] = (
    test_pd_v2_rec["prob_v2"] *
    (test_pd_v2_rec["valor_base"] - test_pd_v2_rec["discount_value"])
)

# Faixas de propensao
test_pd_v2_rec["faixa_propensao"] = pd.cut(
    test_pd_v2_rec["prob_v2"],
    bins=[0, 0.30, 0.55, 0.75, 1.0],
    labels=["Baixa (< 30%)", "Media (30-55%)", "Alta (55-75%)", "Muito alta (> 75%)"]
)

# Tabela resumo por faixa
resumo = test_pd_v2_rec.groupby("faixa_propensao", observed=True).agg(
    n_clientes=("prob_v2", "count"),
    prob_media=("prob_v2", "mean"),
    ticket_medio=("ticket_medio_anterior", "mean"),
    ticket_mediana=("ticket_medio_anterior", "median"),
    freq_media=("freq_anterior", "mean"),
    recencia_media=("recencia", lambda x: x[x < 999].mean()),
    gasto_total_medio=("gasto_total_anterior", "mean"),
    taxa_view_hist=("taxa_view_historica", "mean"),
    taxa_conclusao_hist=("taxa_conclusao_historica", "mean"),
    ticket_ratio_medio=("ticket_ratio", "mean"),
    dias_cadastro_medio=("days_since_registration", "mean"),
    credito_medio=("credit_card_limit", "mean"),
    valor_liquido_medio=("valor_liquido_est", "mean"),
    conversoes_obs=("successful_response", "sum")
).reset_index()

resumo["pct_base"] = (resumo["n_clientes"] / resumo["n_clientes"].sum() * 100).round(1)
resumo["taxa_conversao_obs"] = (resumo["conversoes_obs"] / resumo["n_clientes"] * 100).round(1)

print("=== Tabela Resumo por Faixa de Propensao ===")
cols_exibir = [
    "faixa_propensao", "n_clientes", "pct_base",
    "prob_media", "taxa_conversao_obs",
    "ticket_medio", "ticket_mediana",
    "freq_media", "recencia_media",
    "gasto_total_medio", "taxa_conclusao_hist",
    "ticket_ratio_medio", "dias_cadastro_medio",
    "credito_medio", "valor_liquido_medio"
]
print(resumo[cols_exibir].round(2).to_string(index=False))

In [0]:
# ============================================================
# Visualizacao dos perfis por faixa de propensao
# ============================================================

fig = plt.figure(figsize=(20, 16))
fig.suptitle("Perfil dos Públicos por Faixa de Propensão",
             fontsize=16, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.4)

faixas = ["Baixa (< 30%)", "Media (30-55%)", "Alta (55-75%)", "Muito alta (> 75%)"]
cores = ["#E63946", "#F4A261", "#2A6CF0", "#1D3557"]

# 1. Distribuicao de probabilidade por faixa
ax1 = fig.add_subplot(gs[0, 0])
for faixa, cor in zip(faixas, cores):
    subset = test_pd_v2_rec[test_pd_v2_rec["faixa_propensao"] == faixa]["prob_v2"]
    ax1.hist(subset, bins=20, alpha=0.6, color=cor,
             label=faixa.split(" ")[0], density=True)
ax1.set_title("Distribuição de Probabilidade\npor Faixa", fontweight="bold")
ax1.set_xlabel("P(resposta)")
ax1.set_ylabel("Densidade")
ax1.legend(fontsize=7)

# 2. Ticket medio por faixa
ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(resumo["faixa_propensao"].str.split(" ").str[0],
        resumo["ticket_medio"].round(1),
        color=cores, edgecolor="white", alpha=0.9)
for i, v in enumerate(resumo["ticket_medio"].round(1)):
    ax2.text(i, v + 0.3, f"R${v:.1f}", ha="center", fontsize=8)
ax2.set_title("Ticket Médio Anterior\npor Faixa", fontweight="bold")
ax2.set_ylabel("R$")
ax2.set_xticklabels(resumo["faixa_propensao"].str.split(" ").str[0], fontsize=8)

# 3. Frequencia media por faixa
ax3 = fig.add_subplot(gs[0, 2])
ax3.bar(resumo["faixa_propensao"].str.split(" ").str[0],
        resumo["freq_media"].round(1),
        color=cores, edgecolor="white", alpha=0.9)
for i, v in enumerate(resumo["freq_media"].round(1)):
    ax3.text(i, v + 0.05, f"{v:.1f}", ha="center", fontsize=8)
ax3.set_title("Frequência Média de Compras\npor Faixa", fontweight="bold")
ax3.set_ylabel("Transações anteriores")
ax3.set_xticklabels(resumo["faixa_propensao"].str.split(" ").str[0], fontsize=8)

# 4. Boxplot ticket ratio por faixa
ax4 = fig.add_subplot(gs[1, 0])
dados_box = [
    test_pd_v2_rec[test_pd_v2_rec["faixa_propensao"] == f]["ticket_ratio"].clip(0, 10)
    for f in faixas
]
bp = ax4.boxplot(dados_box, patch_artist=True,
                 tick_labels=["Baixa", "Media", "Alta", "Muito\nAlta"])
for patch, cor in zip(bp["boxes"], cores):
    patch.set_facecolor(cor)
    patch.set_alpha(0.7)
ax4.set_title("Ticket Ratio por Faixa\n(ticket_medio / min_value, cap=10)",
              fontweight="bold")
ax4.set_ylabel("Ticket Ratio")
ax4.axhline(1, color="gray", linestyle="--", linewidth=1, alpha=0.7)

# 5. Dias desde cadastro por faixa
ax5 = fig.add_subplot(gs[1, 1])
ax5.bar(resumo["faixa_propensao"].str.split(" ").str[0],
        resumo["dias_cadastro_medio"].round(0),
        color=cores, edgecolor="white", alpha=0.9)
for i, v in enumerate(resumo["dias_cadastro_medio"].round(0)):
    ax5.text(i, v + 3, f"{int(v)}d", ha="center", fontsize=8)
ax5.set_title("Tempo Médio de Cadastro\npor Faixa (dias)", fontweight="bold")
ax5.set_ylabel("Dias desde cadastro")
ax5.set_xticklabels(resumo["faixa_propensao"].str.split(" ").str[0], fontsize=8)

# 6. Taxa de conclusao historica por faixa
ax6 = fig.add_subplot(gs[1, 2])
ax6.bar(resumo["faixa_propensao"].str.split(" ").str[0],
        (resumo["taxa_conclusao_hist"] * 100).round(1),
        color=cores, edgecolor="white", alpha=0.9)
for i, v in enumerate((resumo["taxa_conclusao_hist"] * 100).round(1)):
    ax6.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=8)
ax6.set_title("Taxa Histórica de Conclusão\npor Faixa", fontweight="bold")
ax6.set_ylabel("Taxa (%)")
ax6.set_xticklabels(resumo["faixa_propensao"].str.split(" ").str[0], fontsize=8)

# 7. Valor liquido estimado por faixa
ax7 = fig.add_subplot(gs[2, 0])
ax7.bar(resumo["faixa_propensao"].str.split(" ").str[0],
        resumo["valor_liquido_medio"].round(2),
        color=cores, edgecolor="white", alpha=0.9)
for i, v in enumerate(resumo["valor_liquido_medio"].round(2)):
    ax7.text(i, v + 0.1, f"R${v:.1f}", ha="center", fontsize=8)
ax7.set_title("Valor Líquido Promocional\nEstimado Médio por Faixa", fontweight="bold")
ax7.set_ylabel("R$")
ax7.set_xticklabels(resumo["faixa_propensao"].str.split(" ").str[0], fontsize=8)

# 8. Credito medio por faixa
ax8 = fig.add_subplot(gs[2, 1])
ax8.bar(resumo["faixa_propensao"].str.split(" ").str[0],
        (resumo["credito_medio"] / 1000).round(1),
        color=cores, edgecolor="white", alpha=0.9)
for i, v in enumerate((resumo["credito_medio"] / 1000).round(1)):
    ax8.text(i, v + 0.3, f"R${v:.0f}k", ha="center", fontsize=8)
ax8.set_title("Limite de Crédito Médio\npor Faixa", fontweight="bold")
ax8.set_ylabel("R$ (milhares)")
ax8.set_xticklabels(resumo["faixa_propensao"].str.split(" ").str[0], fontsize=8)

# 9. Taxa de conversao observada por faixa
ax9 = fig.add_subplot(gs[2, 2])
ax9.bar(resumo["faixa_propensao"].str.split(" ").str[0],
        resumo["taxa_conversao_obs"],
        color=cores, edgecolor="white", alpha=0.9)
for i, v in enumerate(resumo["taxa_conversao_obs"]):
    ax9.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=8)
ax9.set_title("Taxa de Conversão Observada\npor Faixa", fontweight="bold")
ax9.set_ylabel("Taxa (%)")
ax9.set_xticklabels(resumo["faixa_propensao"].str.split(" ").str[0], fontsize=8)

plt.savefig("/Workspace/Users/matheusmartinez2018@gmail.com/coupon-optimization-ml/data/processed/perfil_publicos.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva.")

In [0]:
# ============================================================
# Arvore de decisao rasa — traducao das regras em linguagem
# de negocio. Nao substitui o LightGBM, serve apenas para
# explicar de forma interpretavel o que diferencia os grupos
# ============================================================

from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.preprocessing import LabelEncoder

# Target: faixa de propensao (baixa=0, media=1, alta=2, muito alta=3)
le = LabelEncoder()
y_faixa = le.fit_transform(test_pd_v2_rec["faixa_propensao"].astype(str))

# Features mais interpretaveis para a arvore
FEATURES_ARVORE = [
    "days_since_registration",
    "ticket_medio_anterior",
    "freq_anterior",
    "recencia",
    "taxa_conclusao_historica",
    "ticket_ratio",
    "credit_card_limit"
]

X_arvore = test_pd_v2_rec[FEATURES_ARVORE].fillna(0)

tree = DecisionTreeClassifier(max_depth=3, random_state=42, min_samples_leaf=100)
tree.fit(X_arvore, y_faixa)

print("=== Arvore de Decisao Rasa — Regras Explicativas ===")
print(export_text(tree, feature_names=FEATURES_ARVORE))
print(f"Acuracia da arvore (aproximacao): {tree.score(X_arvore, y_faixa):.2f}")

# Tabela final de publicos para negocio
print("\n=== Tabela Final de Publicos por Campanha ===")
tabela_negocio = pd.DataFrame({
    "Campanha": [
        "Muito Alta Propensao (>75%)",
        "Alta Propensao (55-75%)",
        "Media Propensao (30-55%)",
        "Baixa Propensao (<30%)"
    ],
    "N_clientes": [1174, 2196, 2955, 3885],
    "Pct_base": ["11.5%", "21.5%", "28.9%", "38.1%"],
    "Taxa_conversao_obs": ["77.5%", "62.5%", "40.9%", "11.7%"],
    "Ticket_medio": ["R$17.3", "R$19.3", "R$14.3", "R$4.7"],
    "Freq_media": ["4.5", "4.5", "4.0", "3.9"],
    "Dias_cadastro": ["617d", "379d", "272d", "246d"],
    "Taxa_conclusao_hist": ["51.3%", "54.6%", "40.5%", "16.4%"],
    "Valor_liq_est": ["R$12.7", "R$9.0", "R$5.0", "R$0.9"],
    "Acao_recomendada": [
        "Prioridade maxima — enviar sempre",
        "Alta prioridade — enviar com threshold 0.55",
        "Prioridade media — avaliar custo-beneficio",
        "Nao enviar — baixo retorno esperado"
    ]
})

print(tabela_negocio.to_string(index=False))

In [0]:
# ============================================================
# Arvore de decisao em imagem
# ============================================================

from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(20, 10))

plot_tree(
    tree,
    feature_names=FEATURES_ARVORE,
    class_names=le.classes_,
    filled=True,
    rounded=True,
    fontsize=10,
    ax=ax,
    impurity=False,
    proportion=False
)

ax.set_title("Regras Explicativas — Árvore de Decisão Rasa\n(profundidade 3, apenas para interpretação)",
             fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("/Workspace/Users/matheusmartinez2018@gmail.com/coupon-optimization-ml/data/processed/arvore_explicativa.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Arvore salva.")

In [0]:
# ============================================================
# Re-treinamento com days_since_registration corrigido (v3)
# ============================================================

X_train_v3 = train_pd_v2[FEATURES_V2].copy()
y_train_v3 = train_pd_v2[TARGET].copy()
X_val_v3   = val_pd_v2[FEATURES_V2].copy()
y_val_v3   = val_pd_v2[TARGET].copy()
X_test_v3  = test_pd_v2[FEATURES_V2].copy()
y_test_v3  = test_pd_v2[TARGET].copy()

model_v3 = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    verbose=-1
)

model_v3.fit(
    X_train_v3, y_train_v3,
    eval_set=[(X_val_v3, y_val_v3)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]
)

print("Melhor iteracao:", model_v3.best_iteration_)

resultados_v3 = []
resultados_v3.append(avaliar(model_v2, X_val_v2, y_val_v2, "v2 Validacao (days_reg errado)"))
resultados_v3.append(avaliar(model_v2, X_test_v2, y_test_v2, "v2 Teste    (days_reg errado)"))
resultados_v3.append(avaliar(model_v3, X_val_v3, y_val_v3, "v3 Validacao (days_reg correto)"))
resultados_v3.append(avaliar(model_v3, X_test_v3, y_test_v3, "v3 Teste    (days_reg correto)"))

print("\n=== Comparacao v2 vs v3 ===")
print(pd.DataFrame(resultados_v3).to_string(index=False))